<div style="background: linear-gradient(135deg, #2b2d42 0%, #5c5f8a 100%); color: white; padding: 28px 32px; border-radius: 10px;">
<h1 style="margin: 0; color: white;">Ordinal Classification with Neural Networks</h1>
<p style="margin: 8px 0 0 0; font-size: 1.1em; opacity: 0.9;">Predicting 1–5 star hotel review ratings while respecting the order between classes</p>
<p style="margin: 14px 0 0 0; font-size: 0.9em; opacity: 0.75;">Carles Trullas</p>
</div>

Many classification targets have a natural order: star ratings, disease severity, credit grades, education levels. A standard softmax classifier ignores that order completely. For cross-entropy, predicting 1 star for a 5-star review is exactly as wrong as predicting 4 stars, and nothing in the model ties neighbouring classes together.

This notebook compares three ways of modelling a 5-level ordinal target on hotel reviews, all built on the same small PyTorch network:

| Model | Output | Order-aware? | Consistent by construction? |
|---|---|---|---|
| **Standard classifier** | 5 softmax scores | No | — |
| **Ordinal (cumulative binary)** | 4 independent sigmoids $P(y > k)$ | Yes | No |
| **Rank-consistent ordinal** | 1 shared score + 4 ordered thresholds | Yes | Yes |

### Contents

1. [The ordinal formulation](#1.-The-ordinal-formulation)
2. [Setup](#2.-Setup)
3. [Data](#3.-Data)
4. [Models](#4.-Models)
5. [Training and evaluation](#5.-Training-and-evaluation)
6. [Results](#6.-Results)
7. [Takeaways](#7.-Takeaways)

## 1. The ordinal formulation

With $K$ ordered classes $\{0, 1, \dots, K-1\}$, an ordinal model replaces the single $K$-way decision with $K-1$ binary questions:

$$
\text{Is } y > 0\,?\quad \text{Is } y > 1\,?\quad \dots \quad \text{Is } y > K-2\,?
$$

Each label becomes a "thermometer" vector. With $K = 5$:

| Rating | Class $y$ | Binary target $[y>0,\ y>1,\ y>2,\ y>3]$ |
|---|---|---|
| ★ | 0 | `[0, 0, 0, 0]` |
| ★★ | 1 | `[1, 0, 0, 0]` |
| ★★★ | 2 | `[1, 1, 0, 0]` |
| ★★★★ | 3 | `[1, 1, 1, 0]` |
| ★★★★★ | 4 | `[1, 1, 1, 1]` |

The network outputs $p_k = P(y > k)$ for each $k$, it is trained with binary cross-entropy on every position, and the predicted class is the number of questions answered "yes":

$$
\hat{y} = \sum_{k=0}^{K-2} \mathbb{1}\left[p_k > 0.5\right].
$$

Because a mistake of several classes means several wrong binary answers, larger errors are penalised more than small ones, which is exactly the behaviour we want for ratings.

### The consistency problem

The probabilities must satisfy $p_0 \ge p_1 \ge \dots \ge p_{K-2}$: a review cannot be more likely to be above 3 stars than above 2 stars. If every $p_k$ comes from its own independent output neuron, **nothing enforces this**, and the model can output nonsense such as $[0.9, 0.2, 0.7, 0.1]$.

The rank-consistent model fixes this by construction. It computes **one** score $s(x)$ per sample and compares it against $K-1$ thresholds that are forced to be decreasing,

$$
p_k = \sigma\!\left(s(x) + b_k\right), \qquad b_0 > b_1 > \dots > b_{K-2},
$$

so the cumulative probabilities are always ordered, whatever the weights.

## 2. Setup

In [ ]:
# Uncomment if PyTorch is not installed in your environment
# %pip install torch

In [ ]:
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "data"

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

MODEL_COLORS = {
    "Standard": "#8d99ae",
    "Ordinal": "#e07a3f",
    "Rank-consistent": "#2f6f8f",
}

print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 3. Data

The dataset contains **7,105 hotel reviews**, each with a rating from 1 to 5 stars, perfectly balanced across the five classes. Every review comes with a precomputed **768-dimensional text embedding**, which is what the models use as input. Working on fixed embeddings keeps the focus on the classification head rather than on the text encoder.

- `data/review_data.pickle`: a DataFrame with the `Review` text and its `Rating`.
- `data/embeddings.pickle`: a NumPy array of shape `(7105, 768)`, aligned row by row with the reviews.

In [ ]:
reviews = pd.read_pickle(f"{DATA_DIR}/review_data.pickle")

with open(f"{DATA_DIR}/embeddings.pickle", "rb") as f:
    embeddings = pickle.load(f)

print(f"Reviews: {reviews.shape}  |  Embeddings: {embeddings.shape}")
reviews.head()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))

counts = reviews["Rating"].value_counts().sort_index()
ax1.bar(counts.index.astype(str), counts.values, color="#5c5f8a")
ax1.set(xlabel="Rating (stars)", ylabel="Number of reviews", title="Class distribution")

words = reviews["Review"].str.split().str.len()
ax2.hist(words.clip(upper=words.quantile(0.99)), bins=40, color="#5c5f8a", alpha=0.85)
ax2.set(xlabel="Words per review (clipped at the 99th percentile)", ylabel="Count",
        title=f"Review length (median: {int(words.median())} words)")

plt.tight_layout()
plt.show()

### Train / test split

Ratings are shifted from 1–5 to classes 0–4, and the split is stratified so both sets keep the balanced class distribution.

In [ ]:
NUM_CLASSES = 5

X = embeddings.astype(np.float32)
y = reviews["Rating"].to_numpy() - 1  # 1..5 stars -> classes 0..4

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

X_train_t, X_test_t = torch.from_numpy(X_train), torch.from_numpy(X_test)
y_train_t, y_test_t = torch.from_numpy(y_train).long(), torch.from_numpy(y_test).long()

print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")

### Ordinal label encoding

The "thermometer" transformation from section 1: class $y$ becomes $y$ ones followed by zeros.

In [ ]:
def to_ordinal_targets(y, num_classes):
    # Class y -> [y > 0, y > 1, ..., y > num_classes - 2] as floats
    thresholds = torch.arange(num_classes - 1, device=y.device)
    return (y.unsqueeze(1) > thresholds).float()


to_ordinal_targets(torch.arange(NUM_CLASSES), NUM_CLASSES)

## 4. Models

All three models share the same feature extractor (two hidden layers with ReLU) so that any difference in performance comes from the output head and the loss alone.

In [ ]:
class Backbone(nn.Module):
    # Shared feature extractor: two hidden layers with ReLU.

    def __init__(self, input_size, hidden_size, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

### Standard classifier

$K$ logits, softmax, cross-entropy. The prediction is the class with the highest score.

In [ ]:
class StandardClassifier(nn.Module):

    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.backbone = Backbone(input_size, hidden_size)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))  # logits, shape (n, K)

    def loss(self, logits, y):
        return F.cross_entropy(logits, y)

    @torch.no_grad()
    def predict(self, x):
        return self.forward(x).argmax(dim=1)

### Ordinal classifier (cumulative binary)

$K-1$ independent output neurons, each trained with binary cross-entropy to answer "is $y > k$?". The model is order-aware through its targets, but the four outputs are free to contradict each other.

In [ ]:
class OrdinalClassifier(nn.Module):

    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.backbone = Backbone(input_size, hidden_size)
        self.head = nn.Linear(hidden_size, num_classes - 1)

    def forward(self, x):
        return self.head(self.backbone(x))  # logits of P(y > k), shape (n, K-1)

    def loss(self, logits, y):
        return F.binary_cross_entropy_with_logits(logits, to_ordinal_targets(y, self.num_classes))

    @torch.no_grad()
    def cumulative_proba(self, x):
        return torch.sigmoid(self.forward(x))

    @torch.no_grad()
    def predict(self, x):
        return (self.cumulative_proba(x) > 0.5).sum(dim=1)

### Rank-consistent ordinal classifier

A single score $s(x)$ shared by all thresholds, plus $K-1$ biases. The biases are parametrised as a starting value minus a cumulative sum of positive steps (via `softplus`), which makes them strictly decreasing. As a result $p_0 \ge p_1 \ge \dots$ holds for every input, so the predictions are always consistent.

In [ ]:
class RankConsistentOrdinalClassifier(OrdinalClassifier):

    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__(input_size, hidden_size, num_classes)
        self.head = nn.Linear(hidden_size, 1, bias=False)  # one shared score
        self.first_bias = nn.Parameter(torch.tensor(1.0))
        self.raw_steps = nn.Parameter(torch.zeros(num_classes - 2))

    @property
    def biases(self):
        # b_0 > b_1 > ... > b_{K-2}: each step down is softplus(.) > 0
        steps = F.softplus(self.raw_steps)
        return torch.cat([self.first_bias.view(1), self.first_bias - torch.cumsum(steps, dim=0)])

    def forward(self, x):
        score = self.head(self.backbone(x))  # (n, 1)
        return score + self.biases           # broadcast to (n, K-1)

## 5. Training and evaluation

All models are trained with exactly the same loop and settings: Adam, learning rate $10^{-3}$, batch size 64, 25 epochs, shuffled mini-batches.

Besides accuracy, the evaluation reports metrics that matter for ordered targets:

- **MAE (stars):** mean absolute distance between predicted and true rating. Predicting 4★ for a 5★ review costs 1, predicting 1★ costs 4.
- **Within ±1 star:** share of predictions at most one star away from the truth.
- **Inconsistent rows:** for the ordinal models, share of test samples whose cumulative probabilities are not non-increasing.

In [ ]:
def inconsistency_rate(model, X):
    # Share of samples where P(y > k) increases somewhere along k
    if not hasattr(model, "cumulative_proba"):
        return np.nan
    p = model.cumulative_proba(X)
    return (p[:, 1:] > p[:, :-1] + 1e-6).any(dim=1).float().mean().item()


@torch.no_grad()
def evaluate(model, X, y):
    model.eval()
    X, y = X.to(DEVICE), y.to(DEVICE)
    pred = model.predict(X)
    err = (pred - y).abs().float()
    return {
        "Accuracy": (err == 0).float().mean().item(),
        "MAE (stars)": err.mean().item(),
        "Within ±1 star": (err <= 1).float().mean().item(),
        "Inconsistent rows": inconsistency_rate(model, X),
    }


def train(model, epochs=25, batch_size=64, lr=1e-3):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)

    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = model.loss(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(xb)

        metrics = evaluate(model, X_test_t, y_test_t)
        history.append({"epoch": epoch, "train_loss": running_loss / len(X_train_t), **metrics})

    return pd.DataFrame(history)

In [ ]:
INPUT_SIZE, HIDDEN_SIZE = X_train.shape[1], 64

model_classes = {
    "Standard": StandardClassifier,
    "Ordinal": OrdinalClassifier,
    "Rank-consistent": RankConsistentOrdinalClassifier,
}

models, histories = {}, {}
for name, cls in model_classes.items():
    torch.manual_seed(SEED)  # same initial backbone and batch order for every model
    models[name] = cls(INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES)
    histories[name] = train(models[name])
    last = histories[name].iloc[-1]
    print(f"{name:>16} | test accuracy {last['Accuracy']:.3f} | test MAE {last['MAE (stars)']:.3f}")

## 6. Results

### Summary table

Final test-set metrics after training. For accuracy and within ±1 higher is better; for MAE and inconsistency lower is better.

In [ ]:
results = pd.DataFrame(
    {name: evaluate(model, X_test_t, y_test_t) for name, model in models.items()}
).T

(results.style
    .format({"Accuracy": "{:.1%}", "MAE (stars)": "{:.3f}",
             "Within ±1 star": "{:.1%}", "Inconsistent rows": "{:.2%}"}, na_rep="—")
    .highlight_max(subset=["Accuracy", "Within ±1 star"], color="#d8ecd8")
    .highlight_min(subset=["MAE (stars)"], color="#d8ecd8"))

### Learning curves

Test accuracy and test MAE after each epoch.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

for name, hist in histories.items():
    ax1.plot(hist["epoch"], hist["Accuracy"], lw=2, color=MODEL_COLORS[name], label=name)
    ax2.plot(hist["epoch"], hist["MAE (stars)"], lw=2, color=MODEL_COLORS[name], label=name)

ax1.set(xlabel="Epoch", ylabel="Test accuracy", title="Accuracy (higher is better)")
ax2.set(xlabel="Epoch", ylabel="Test MAE (stars)", title="Mean absolute error (lower is better)")
ax1.legend(frameon=False)

plt.tight_layout()
plt.show()

### Confusion matrices

Rows are true ratings, columns are predicted ratings, each row normalised to sum to 100%. An order-aware model should concentrate its errors next to the diagonal: confusing 4★ with 5★ is far more acceptable than confusing 1★ with 5★.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
stars = [f"{i}★" for i in range(1, NUM_CLASSES + 1)]

for ax, (name, model) in zip(axes, models.items()):
    model.eval()
    pred = model.predict(X_test_t.to(DEVICE)).cpu().numpy()
    cm = confusion_matrix(y_test, pred, labels=range(NUM_CLASSES), normalize="true")

    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cm[i, j]:.0%}", ha="center", va="center",
                    color="white" if cm[i, j] > 0.5 else "#333", fontsize=10)
    ax.set_xticks(range(NUM_CLASSES), stars)
    ax.set_yticks(range(NUM_CLASSES), stars)
    ax.set(xlabel="Predicted", ylabel="True", title=name)
    ax.grid(False)

plt.tight_layout()
plt.show()

### Size of the errors

The distribution of $|\hat{y} - y|$ for each model: the share of predictions that are exact, off by one star, off by two, and so on.

In [ ]:
error_dist = pd.DataFrame({
    name: pd.Series(np.abs(model.predict(X_test_t.to(DEVICE)).cpu().numpy() - y_test))
          .value_counts(normalize=True)
          .reindex(range(NUM_CLASSES), fill_value=0)
    for name, model in models.items()
})
error_dist.index = [f"{d} star{'s' if d != 1 else ''} off" if d else "Exact" for d in error_dist.index]

ax = error_dist.plot.bar(figsize=(10, 4), color=[MODEL_COLORS[c] for c in error_dist.columns],
                         width=0.8, rot=0)
ax.set(ylabel="Share of test predictions", title="Distribution of absolute errors")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Consistency of the cumulative probabilities

For a few test reviews, the predicted $P(y > k)$ of both ordinal models. A consistent curve only goes down; any upward step is a contradiction. The examples shown are the ones where the unconstrained ordinal model is most inconsistent.

In [ ]:
ordinal, consistent = models["Ordinal"], models["Rank-consistent"]
X_dev = X_test_t.to(DEVICE)
p_ord = ordinal.cumulative_proba(X_dev).cpu().numpy()
p_con = consistent.cumulative_proba(X_dev).cpu().numpy()

# Largest upward jump along k for the unconstrained model
violation = np.clip(np.diff(p_ord, axis=1), 0, None).max(axis=1)
examples = np.argsort(-violation)[:4]

fig, axes = plt.subplots(1, len(examples), figsize=(15, 3.6), sharey=True)
ks = [f"> {k + 1}★" for k in range(NUM_CLASSES - 1)]

for ax, i in zip(axes, examples):
    ax.plot(ks, p_ord[i], "o-", lw=2, color=MODEL_COLORS["Ordinal"], label="Ordinal")
    ax.plot(ks, p_con[i], "s-", lw=2, color=MODEL_COLORS["Rank-consistent"], label="Rank-consistent")
    ax.axhline(0.5, color="grey", lw=1, ls=":")
    ax.set(title=f"True rating: {y_test[i] + 1}★", ylim=(-0.05, 1.05))

axes[0].set_ylabel("P(rating > threshold)")
axes[0].legend(frameon=False, loc="lower left")
plt.tight_layout()
plt.show()

print(f"Inconsistent test rows | Ordinal: {inconsistency_rate(ordinal, X_dev):.2%} "
      f"| Rank-consistent: {inconsistency_rate(consistent, X_dev):.2%}")

## 7. Takeaways

- **Accuracy alone hides what matters for ordered targets.** Two models with similar accuracy can behave very differently in *how* they are wrong. MAE, the within-±1 rate and the confusion matrices show whether errors fall next to the true class or far from it.
- **The ordinal formulation encodes order through the targets.** Turning one 5-way decision into four cumulative "is it above $k$?" questions makes larger mistakes cost more during training, at no extra architectural cost.
- **Independent ordinal heads can contradict themselves.** Nothing forces $P(y>k)$ to decrease with $k$ when each threshold has its own neuron, so the predicted label can come from an incoherent set of probabilities.
- **Sharing a single score with ordered thresholds removes the problem by construction.** The rank-consistent model guarantees monotone cumulative probabilities for any input, and it also has fewer output parameters.
- **Middle ratings are the hardest.** 1★ and 5★ reviews are usually unambiguous, while 2★–4★ reviews mix positive and negative points. Here the ordinal models are expected to help most, by keeping their mistakes within one star.